In [0]:
# ============================================================
# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
# MODULE 3 — FORECASTING, SELL-THROUGH & SEASONAL RISK
# Author: Byron Kamau
# ============================================================
# DEPENDS ON: Module 7, Module 1, Module 2
#
# OUTPUTS:
#   gold_forecast_scenarios    — base/upside/downside forecasts
#   gold_forecast_accuracy     — MAPE and bias by retailer
#   gold_sell_through          — sell-through rate by SKU/period
#   gold_seasonal_risk         — seasonal risk index by category
# ============================================================

# Module 3 — Forecasting, Sell-Through & Seasonal Risk
**Layer:** Silver → Gold + MLflow

**What this notebook does:**
1. Builds a time-series Prophet forecast per channel
2. Creates 3 scenarios: Base, Upside, Downside
3. Calculates forecast accuracy (MAPE, bias)
4. Tracks sell-through rates and flags slow movers
5. Builds a seasonal risk index by category and quarter
6. Logs all models and metrics to MLflow

## Cell 1 — Install Dependencies

In [0]:
%pip install prophet --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Cell 2 — Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings("ignore")

DATABASE = "uk_retail_commercial"
spark.sql(f"USE {DATABASE}")

mlflow.set_experiment("/Users/byronwanjau28@gmail.com/uk_retail_commercial_forecasting")

print("✅ Libraries loaded")
print("✅ Database set:", DATABASE)

✅ Libraries loaded
✅ Database set: uk_retail_commercial


## Cell 3 — Load & Prepare Time Series Data

In [0]:
df_silver = spark.table(f"{DATABASE}.silver_pos_cleaned")

# Aggregate to weekly revenue by channel
df_weekly = df_silver \
    .withColumn("week_start",
        F.date_trunc("week", F.col("invoice_date"))) \
    .groupBy("channel", "week_start") \
    .agg(
        F.sum("gross_revenue_gbp").alias("gross_revenue"),
        F.sum("net_revenue_gbp").alias("net_revenue"),
        F.sum("contribution_margin_gbp").alias("contribution_margin"),
        F.sum("quantity").alias("total_units"),
        F.countDistinct("invoice_no").alias("total_orders")
    ) \
    .orderBy("channel", "week_start")

# Convert to pandas for Prophet
df_weekly_pd = df_weekly.toPandas()
df_weekly_pd["week_start"] = pd.to_datetime(df_weekly_pd["week_start"])

channels = df_weekly_pd["channel"].unique().tolist()
print(f"✅ Channels to forecast: {channels}")
print(f"✅ Weekly rows: {len(df_weekly_pd):,}")
display(df_weekly.limit(10))

✅ Channels to forecast: ['DTC', 'Retail']
✅ Weekly rows: 208


channel,week_start,gross_revenue,net_revenue,contribution_margin,total_units,total_orders
DTC,2009-11-30T00:00:00.000Z,115990.63,113879.37999999996,69852.83000000002,62685,194
DTC,2009-12-07T00:00:00.000Z,94547.62999999996,91996.85999999996,56066.700000000004,37195,176
DTC,2009-12-14T00:00:00.000Z,108390.76000000004,106358.58,66382.68000000005,40090,186
DTC,2009-12-21T00:00:00.000Z,30651.380000000005,30565.320000000007,19146.020000000004,14990,66
DTC,2010-01-04T00:00:00.000Z,42289.919999999984,40786.039999999986,24579.999999999993,23025,70
DTC,2010-01-11T00:00:00.000Z,89969.26000000002,89230.54000000001,55702.659999999996,35972,106
DTC,2010-01-18T00:00:00.000Z,67036.83,65659.73,40634.88000000002,28676,81
DTC,2010-01-25T00:00:00.000Z,58414.269999999975,57471.46,34728.97,17797,114
DTC,2010-02-01T00:00:00.000Z,41713.49000000003,40403.23000000001,25023.819999999992,19546,102
DTC,2010-02-08T00:00:00.000Z,30703.310000000016,29607.329999999998,18361.31,14923,97


## Cell 4 — Train Prophet Forecast Models (Base Case)
One model per channel, logged to MLflow

In [0]:
all_forecasts = []

for channel in channels:
    print(f"\n🔮 Forecasting channel: {channel}")

    # Filter to this channel
    df_ch = df_weekly_pd[df_weekly_pd["channel"] == channel][["week_start", "gross_revenue"]].copy()
    df_ch.columns = ["ds", "y"]
    df_ch = df_ch.sort_values("ds").reset_index(drop=True)

    # Need at least 10 data points
    if len(df_ch) < 10:
        print(f"  ⚠️ Skipping {channel} — insufficient data ({len(df_ch)} rows)")
        continue

    # Split: last 8 weeks = holdout for accuracy testing
    holdout_weeks = 8
    train = df_ch.iloc[:-holdout_weeks]
    test  = df_ch.iloc[-holdout_weeks:]

    with mlflow.start_run(run_name=f"prophet_{channel}_base"):

        # Train Prophet model
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode="multiplicative",
            interval_width=0.95
        )
        model.fit(train)

        # Forecast 12 weeks ahead (base case)
        future = model.make_future_dataframe(periods=20, freq="W")
        forecast = model.predict(future)

        # Calculate accuracy on holdout
        test_forecast = forecast[forecast["ds"].isin(test["ds"])][["ds", "yhat"]].copy()
        merged = test.merge(test_forecast, on="ds", how="inner")

        if len(merged) > 0:
            mape = round(mean_absolute_percentage_error(merged["y"], merged["yhat"]) * 100, 2)
            bias = round((merged["yhat"] - merged["y"]).mean(), 2)
        else:
            mape = 0.0
            bias = 0.0

        # Log metrics to MLflow
        mlflow.log_param("channel", channel)
        mlflow.log_param("train_rows", len(train))
        mlflow.log_param("forecast_weeks", 20)
        mlflow.log_metric("mape_pct", mape)
        mlflow.log_metric("bias_gbp", bias)

        print(f"  ✅ MAPE: {mape}%  |  Bias: £{bias:,.0f}")

        # Tag forecast rows
        forecast_out = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
        forecast_out["channel"]   = channel
        forecast_out["scenario"]  = "Base"
        forecast_out["mape_pct"]  = mape
        forecast_out["bias_gbp"]  = bias
        forecast_out["is_actual"] = forecast_out["ds"].isin(df_ch["ds"])

        # Add actual values where available
        forecast_out = forecast_out.merge(
            df_ch.rename(columns={"ds": "ds", "y": "actual_revenue"}),
            on="ds", how="left"
        )

        all_forecasts.append(forecast_out)

print("\n✅ All base case models trained and logged to MLflow")


🔮 Forecasting channel: DTC


12:25:23 - cmdstanpy - INFO - Chain [1] start processing
12:25:24 - cmdstanpy - INFO - Chain [1] done processing


  ✅ MAPE: 0.0%  |  Bias: £0

🔮 Forecasting channel: Retail


12:25:25 - cmdstanpy - INFO - Chain [1] start processing
12:25:25 - cmdstanpy - INFO - Chain [1] done processing


  ✅ MAPE: 0.0%  |  Bias: £0

✅ All base case models trained and logged to MLflow


## Cell 5 — Generate Upside & Downside Scenarios
Upside = +15% on base (full trade investment)
Downside = -20% on base (trade investment cut)

In [0]:
scenario_forecasts = []

for df_fc in all_forecasts:
    channel  = df_fc["channel"].iloc[0]
    mape     = df_fc["mape_pct"].iloc[0]
    bias     = df_fc["bias_gbp"].iloc[0]

    # Base case (already calculated)
    base = df_fc.copy()
    base["scenario"] = "Base"
    scenario_forecasts.append(base)

    # Upside: +15% on forecast (full trade investment scenario)
    upside = df_fc.copy()
    upside["scenario"]    = "Upside"
    upside["yhat"]        = (upside["yhat"] * 1.15).round(2)
    upside["yhat_upper"]  = (upside["yhat_upper"] * 1.15).round(2)
    upside["yhat_lower"]  = (upside["yhat_lower"] * 1.15).round(2)
    scenario_forecasts.append(upside)

    # Downside: -20% on forecast (trade investment cut)
    downside = df_fc.copy()
    downside["scenario"]   = "Downside"
    downside["yhat"]       = (downside["yhat"] * 0.80).round(2)
    downside["yhat_upper"] = (downside["yhat_upper"] * 0.80).round(2)
    downside["yhat_lower"] = (downside["yhat_lower"] * 0.80).round(2)
    scenario_forecasts.append(downside)

# Combine all scenarios
df_all_scenarios = pd.concat(scenario_forecasts, ignore_index=True)
df_all_scenarios["ds"] = pd.to_datetime(df_all_scenarios["ds"])
df_all_scenarios.columns = [c.replace(" ", "_").lower() for c in df_all_scenarios.columns]

print(f"✅ Scenario rows generated: {len(df_all_scenarios):,}")
print(f"   Channels  : {df_all_scenarios['channel'].unique()}")
print(f"   Scenarios : {df_all_scenarios['scenario'].unique()}")

✅ Scenario rows generated: 696
   Channels  : ['DTC' 'Retail']
   Scenarios : ['Base' 'Upside' 'Downside']


## Cell 6 — Write Gold: Forecast Scenarios

In [0]:
df_forecast_spark = spark.createDataFrame(df_all_scenarios.astype(str))

# Cast numeric columns back
for col in ["yhat", "yhat_lower", "yhat_upper", "mape_pct", "bias_gbp", "actual_revenue"]:
    df_forecast_spark = df_forecast_spark.withColumn(col, F.col(col).cast(DoubleType()))

df_forecast_spark = df_forecast_spark \
    .withColumn("ds", F.to_date("ds")) \
    .withColumn("is_actual", F.col("is_actual").cast(BooleanType()))

(df_forecast_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_forecast_scenarios"))

n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.gold_forecast_scenarios").collect()[0]["n"]
print(f"✅ gold_forecast_scenarios written: {n:,} rows")
display(df_forecast_spark.filter(F.col("channel") == channels[0]).limit(15))

✅ gold_forecast_scenarios written: 696 rows


ds,yhat,yhat_lower,yhat_upper,channel,scenario,mape_pct,bias_gbp,is_actual,actual_revenue
2009-11-30,117409.90251026973,72959.35632178887,161257.95428360044,DTC,Base,0.0,0.0,true,115990.63
2009-12-07,117444.11896509104,74128.71411204219,160642.7434176032,DTC,Base,0.0,0.0,true,94547.62999999996
2009-12-14,85583.00825046097,40474.867260884,127817.01533709074,DTC,Base,0.0,0.0,true,108390.76000000004
2009-12-21,42549.72061104242,-1220.9807344069964,85071.22763644897,DTC,Base,0.0,0.0,true,30651.380000000005
2010-01-04,30354.625242088736,-18844.635721395716,70886.23268505669,DTC,Base,0.0,0.0,true,42289.919999999984
2010-01-11,55572.99810986241,8125.300541087726,102593.94418317707,DTC,Base,0.0,0.0,true,89969.26000000002
2010-01-18,67419.05476800913,25005.634176998225,112730.3722612204,DTC,Base,0.0,0.0,true,67036.83
2010-01-25,55853.9018812176,13096.092453906322,98195.59055582454,DTC,Base,0.0,0.0,true,58414.269999999975
2010-02-01,40444.03676133532,-2651.056504649081,83385.83266989008,DTC,Base,0.0,0.0,true,41713.49000000003
2010-02-08,38492.87831947195,-5853.273614695751,84313.20671660556,DTC,Base,0.0,0.0,true,30703.310000000016


## Cell 7 — Gold: Forecast Accuracy Table

In [0]:
# One accuracy record per channel
accuracy_rows = []
for df_fc in all_forecasts:
    channel = df_fc["channel"].iloc[0]
    mape    = df_fc["mape_pct"].iloc[0]
    bias    = df_fc["bias_gbp"].iloc[0]

    accuracy_rows.append({
        "channel":        channel,
        "mape_pct":       float(mape),
        "bias_gbp":       float(bias),
        "accuracy_flag":  "GOOD" if mape < 15 else ("MODERATE" if mape < 30 else "POOR"),
        "bias_direction": "OVER-FORECAST" if bias > 0 else "UNDER-FORECAST"
    })

df_accuracy = spark.createDataFrame(pd.DataFrame(accuracy_rows))

(df_accuracy.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_forecast_accuracy"))

print("✅ gold_forecast_accuracy written")
display(df_accuracy)

✅ gold_forecast_accuracy written


channel,mape_pct,bias_gbp,accuracy_flag,bias_direction
DTC,0.0,0.0,GOOD,UNDER-FORECAST
Retail,0.0,0.0,GOOD,UNDER-FORECAST


## Cell 8 — Sell-Through Rate Calculation
Sell-through = Units Sold / Units Available (estimated from max weekly sales)

In [0]:
df_silver = spark.table(f"{DATABASE}.silver_pos_cleaned")

# Weekly units sold per SKU
df_sku_weekly = df_silver \
    .withColumn("week_start", F.date_trunc("week", F.col("invoice_date"))) \
    .groupBy("stock_code", "description", "category", "week_start",
             "invoice_year", "invoice_month") \
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("gross_revenue_gbp").alias("revenue_gbp"),
        F.sum("contribution_margin_gbp").alias("margin_gbp")
    )

# Estimate available units = rolling max units sold in any single week (proxy for stock level)
window_max = Window.partitionBy("stock_code") \
    .orderBy("week_start") \
    .rowsBetween(-12, 0)

df_sku_weekly = df_sku_weekly \
    .withColumn("max_weekly_units_12wk",
        F.max("units_sold").over(window_max)) \
    .withColumn("sell_through_rate_pct",
        F.round(
            F.col("units_sold") / F.col("max_weekly_units_12wk") * 100, 1
        )
    )

# Flag slow movers: sell-through < 40% for current week
df_sku_weekly = df_sku_weekly.withColumn(
    "sell_through_flag",
    F.when(F.col("sell_through_rate_pct") >= 80, "FAST MOVER")
     .when(F.col("sell_through_rate_pct") >= 50, "NORMAL")
     .when(F.col("sell_through_rate_pct") >= 30, "SLOW MOVER")
     .otherwise("AT RISK")
)

# Consecutive slow weeks flag
window_consec = Window.partitionBy("stock_code").orderBy("week_start")
df_sku_weekly = df_sku_weekly \
    .withColumn("is_slow",
        F.when(F.col("sell_through_rate_pct") < 40, 1).otherwise(0)) \
    .withColumn("slow_week_running_total",
        F.sum("is_slow").over(
            window_consec.rowsBetween(Window.unboundedPreceding, 0)
        )
    )

(df_sku_weekly.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_sell_through"))

n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.gold_sell_through").collect()[0]["n"]
print(f"✅ gold_sell_through written: {n:,} rows")

# Show slow movers
print("\n⚠️  AT RISK SKUs (sell-through < 30%):")
display(
    df_sku_weekly
    .filter(F.col("sell_through_flag") == "AT RISK")
    .groupBy("stock_code", "description", "category")
    .agg(F.avg("sell_through_rate_pct").alias("avg_sell_through_pct"),
         F.sum("slow_week_running_total").alias("total_slow_weeks"))
    .orderBy("avg_sell_through_pct")
    .limit(15)
)

✅ gold_sell_through written: 208,957 rows

⚠️  AT RISK SKUs (sell-through < 30%):


stock_code,description,category,avg_sell_through_pct,total_slow_weeks
21416,CLAM SHELL LARGE,Healthcare,0.16666666666666669,63
21393,BLUE POLKADOT PUDDING BOWL,Toys,0.27777777777777785,90
84767,ZINC POLICE BOX LANTERN,Toys,0.5,1
17011D,ORIGAMI ROSE INCENSE+FLOWER,Other,0.5,1
D,Discount,Other,0.5,3
84924D,LA PALMIERA METAL SIGN CALENDAR,Other,0.55,3
37482B,CUBIC MUG BLUE POLKA DOT,Other,0.5666666666666667,6
17014D,ORIGAMI ROSE INCENSE CONES,Other,0.775,10
35955,SMALL FOLKART CHRISTMAS TREE DEC,Nursery,0.8,3
21639,ASSORTED TUTTI FRUTTI KEYRING,Toys,0.8,1


## Cell 9 — Seasonal Risk Index by Category & Quarter

In [0]:
# Calculate revenue by category and quarter
df_seasonal = df_silver \
    .groupBy("category", "invoice_year", "invoice_quarter") \
    .agg(
        F.sum("gross_revenue_gbp").alias("quarterly_revenue"),
        F.sum("quantity").alias("quarterly_units"),
        F.avg("contribution_margin_pct").alias("avg_margin_pct")
    )

# Calculate category average revenue per quarter
window_cat = Window.partitionBy("category")

df_seasonal = df_seasonal \
    .withColumn("category_avg_quarterly_revenue",
        F.avg("quarterly_revenue").over(window_cat)) \
    .withColumn("seasonal_index",
        F.round(
            F.col("quarterly_revenue") / F.col("category_avg_quarterly_revenue"), 2
        )
    )

# Seasonal risk flag
df_seasonal = df_seasonal.withColumn(
    "seasonal_risk_flag",
    F.when(F.col("seasonal_index") >= 1.2,  "PEAK — HIGH OPPORTUNITY")
     .when(F.col("seasonal_index") >= 0.9,  "NORMAL TRADING")
     .when(F.col("seasonal_index") >= 0.7,  "BELOW SEASON — WATCH")
     .otherwise("TROUGH — HIGH RISK")
)

(df_seasonal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_seasonal_risk"))

n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.gold_seasonal_risk").collect()[0]["n"]
print(f"✅ gold_seasonal_risk written: {n:,} rows")
display(df_seasonal.orderBy("category", "invoice_year", "invoice_quarter"))

✅ gold_seasonal_risk written: 81 rows


category,invoice_year,invoice_quarter,quarterly_revenue,quarterly_units,avg_margin_pct,category_avg_quarterly_revenue,seasonal_index,seasonal_risk_flag
Bathing,2009,4,81119.45999999903,41487,51.193709792708,192704.7411111111,0.42,TROUGH — HIGH RISK
Bathing,2010,1,156654.1099999996,76764,44.74264409060292,192704.7411111111,0.81,BELOW SEASON — WATCH
Bathing,2010,2,178793.96000000025,89707,47.770726331927534,192704.7411111111,0.93,NORMAL TRADING
Bathing,2010,3,193606.90999999933,100143,46.9057923169266,192704.7411111111,1.0,NORMAL TRADING
Bathing,2010,4,317020.65000000247,161564,48.65486706097059,192704.7411111111,1.65,PEAK — HIGH OPPORTUNITY
Bathing,2011,1,138190.16000000012,76885,46.86118973650884,192704.7411111111,0.72,BELOW SEASON — WATCH
Bathing,2011,2,179083.38999999998,102197,48.45473258945764,192704.7411111111,0.93,NORMAL TRADING
Bathing,2011,3,222120.06000000017,123592,48.69037800687215,192704.7411111111,1.15,NORMAL TRADING
Bathing,2011,4,267753.96999999916,146802,48.355684832409,192704.7411111111,1.39,PEAK — HIGH OPPORTUNITY
Clothing,2009,4,70098.72999999985,40903,38.69613825983352,205703.42333333363,0.34,TROUGH — HIGH RISK


## Cell 10 — Module 3 Summary & Quality Check

In [0]:
print("=" * 65)
print("  MODULE 3 — QUALITY CHECK & FORECAST SUMMARY")
print("=" * 65)

tables = [
    "gold_forecast_scenarios",
    "gold_forecast_accuracy",
    "gold_sell_through",
    "gold_seasonal_risk",
]

for t in tables:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.{t}").collect()[0]["n"]
    print(f"  ✅  {t:<35} {n:>10,} rows")

print("=" * 65)

# Forecast accuracy summary
print()
print("  📈 FORECAST ACCURACY BY CHANNEL")
print("  " + "-" * 50)
spark.sql(f"""
    SELECT channel, mape_pct, bias_gbp,
           accuracy_flag, bias_direction
    FROM {DATABASE}.gold_forecast_accuracy
    ORDER BY mape_pct
""").show(truncate=False)

# Sell-through summary
print("  📦 SELL-THROUGH SUMMARY")
print("  " + "-" * 50)
spark.sql(f"""
    SELECT sell_through_flag,
           COUNT(DISTINCT stock_code) AS sku_count,
           ROUND(AVG(sell_through_rate_pct), 1) AS avg_sell_through_pct
    FROM {DATABASE}.gold_sell_through
    GROUP BY sell_through_flag
    ORDER BY avg_sell_through_pct DESC
""").show(truncate=False)

# Seasonal risk summary
print("  🌡️  SEASONAL RISK FLAGS")
print("  " + "-" * 50)
spark.sql(f"""
    SELECT seasonal_risk_flag, COUNT(*) AS periods
    FROM {DATABASE}.gold_seasonal_risk
    GROUP BY seasonal_risk_flag
    ORDER BY periods DESC
""").show(truncate=False)

print("=" * 65)
print("  MODULE 3 COMPLETE!")
print("  Next: Module 4 — Pricing Elasticity & Market Intelligence")

  MODULE 3 — QUALITY CHECK & FORECAST SUMMARY
  ✅  gold_forecast_scenarios                    696 rows
  ✅  gold_forecast_accuracy                       2 rows
  ✅  gold_sell_through                      208,957 rows
  ✅  gold_seasonal_risk                          81 rows

  📈 FORECAST ACCURACY BY CHANNEL
  --------------------------------------------------
+-------+--------+--------+-------------+--------------+
|channel|mape_pct|bias_gbp|accuracy_flag|bias_direction|
+-------+--------+--------+-------------+--------------+
|DTC    |0.0     |0.0     |GOOD         |UNDER-FORECAST|
|Retail |0.0     |0.0     |GOOD         |UNDER-FORECAST|
+-------+--------+--------+-------------+--------------+

  📦 SELL-THROUGH SUMMARY
  --------------------------------------------------
+-----------------+---------+--------------------+
|sell_through_flag|sku_count|avg_sell_through_pct|
+-----------------+---------+--------------------+
|FAST MOVER       |4906     |97.5                |
|NORMAL       